In [ ]:
"""
Audit Logs to Bronze Layer - OCI AIDP
CORRECTED with proper volume path
"""

from pyspark.sql.functions import current_timestamp, input_file_name, sha2, col

# Configure Spark
spark.conf.set("spark.sql.files.ignoreMissingFiles", "true")
spark.conf.set("spark.sql.files.ignoreCorruptFiles", "true")

# ============================================================================
# CONFIGURATION - CORRECTED PATHS
# ============================================================================

AUDIT_BASE = "file:/Volumes/security_lake/default/audit_logs"

# CORRECTED - removed /volumes/ from path
BRONZE_BASE = "/Volumes/gitrepo/default/git_oci_aidp_bronze/audit_logs"
BRONZE_DATA_PATH = f"{BRONZE_BASE}/data"
BRONZE_CHECKPOINT = f"{BRONZE_BASE}/checkpoints"

print("=" * 70)
print("AUDIT LOGS TO BRONZE - STRUCTURED STREAMING")
print("=" * 70)
print(f"Source Path: {AUDIT_BASE}")
print(f"Bronze Data Path: {BRONZE_DATA_PATH}")
print(f"Checkpoint Path: {BRONZE_CHECKPOINT}")
print("=" * 70)

# ============================================================================
# STREAM READ
# ============================================================================

print("\n[STEP 1] Setting up stream reader...")

raw_stream = (
    spark.readStream
        .format("text")
        .option("recursiveFileLookup", "true")
        .option("maxFilesPerTrigger", 100)
        .load(AUDIT_BASE)
        .withColumnRenamed("value", "raw_json")
        .withColumn("source_file", input_file_name())
        .withColumn("ingest_time", current_timestamp())
        .withColumn("event_hash", sha2(col("raw_json"), 256))
)

print("✓ Stream reader configured")

# ============================================================================
# WRITE BRONZE
# ============================================================================

print("\n[STEP 2] Starting streaming write to bronze layer...")
print("Processing files in batches...")

query = (
    raw_stream.writeStream
        .format("parquet")
        .outputMode("append")
        .option("checkpointLocation", BRONZE_CHECKPOINT)
        .trigger(availableNow=True)
        .start(BRONZE_DATA_PATH)
)

query.awaitTermination()

print("\n✓ Bronze ingestion complete!")

# ============================================================================
# VERIFICATION
# ============================================================================

print("\n[STEP 3] Verifying bronze layer...")

bronze_df = spark.read.parquet(BRONZE_DATA_PATH)

total_records = bronze_df.count()
unique_files = bronze_df.select("source_file").distinct().count()

print(f"\nBronze Layer Statistics:")
print(f"  Total records: {total_records:,}")
print(f"  Unique source files: {unique_files:,}")

print("\nTop 10 files by record count:")
(bronze_df
    .groupBy("source_file")
    .count()
    .orderBy(col("count").desc())
    .show(10, truncate=False))

print("\nSample records:")
bronze_df.select("raw_json", "source_file", "ingest_time").show(3, truncate=100)

print("\n" + "=" * 70)
print("✓ PROCESSING COMPLETED SUCCESSFULLY")
print("=" * 70)

AUDIT LOGS TO BRONZE - STRUCTURED STREAMING
Source Path: file:/Volumes/security_lake/default/audit_logs
Bronze Data Path: /Volumes/gitrepo/default/git_oci_aidp_bronze/audit_logs/data
Checkpoint Path: /Volumes/gitrepo/default/git_oci_aidp_bronze/audit_logs/checkpoints

[STEP 1] Setting up stream reader...


✓ Stream reader configured

[STEP 2] Starting streaming write to bronze layer...
Processing files in batches...



✓ Bronze ingestion complete!

[STEP 3] Verifying bronze layer...



Bronze Layer Statistics:
  Total records: 21,249,885
  Unique source files: 9,392

Top 10 files by record count:


+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+
|source_file                                                                                                                                                                          |count|
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+
|file:/Volumes/security_lake/default/audit_logs/ocid1.serviceconnector.oc1.iad.amaaaaaac3adhhqafmgsex7eimr6i7if77xbsctxy3wgms2ayabmijg3jlnq/20260103T013802Z_20260103T014723Z.0.log.gz|76010|
|file:/Volumes/security_lake/default/audit_logs/ocid1.serviceconnector.oc1.iad.amaaaaaac3adhhqafmgsex7eimr6i7if77xbsctxy3wgms2ayabmijg3jlnq/20251224T015432Z_20251224T020331Z.0.log.gz|73375|
|file:/Volumes/security_lake/default/audit_logs/oc

+----------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------+-----------------------+
|                                                                                            raw_json|                                                                                         source_file|            ingest_time|
+----------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------+-----------------------+
|{"data":{"additionalDetails":{"X-Real-Port":45437},"availabilityDomain":"AD1","compartmentId":"oc...|file:/Volumes/security_lake/default/audit_logs/ocid1.serviceconnector.oc1.iad.amaaaaaac3adhhqafmg...|2026-02-02 19:53:08.613|
|{"data":{"additionalDetails":{"X-Real-Port":45437},"availabilityDomain":"AD1","compartm

In [4]:
%sql
SELECT 
    TO_DATE(ingest_time) as ingest_date,
    COUNT(*) as records
FROM parquet.`/Volumes/gitrepo/default/git_oci_aidp_bronze/audit_logs/data`
GROUP BY TO_DATE(ingest_time)
ORDER BY ingest_date DESC
LIMIT 10